# 23 — Models, post-processing, MRR@5 and the `predictions` table (Milestone 3)

Owner: Member 3. Input: `data/m3/pairs.parquet` of `22_features` (one row per (labelled entry, reference)
of the same paper, with `label`, `source`, `partition` and the `f_*` features). Output:
`logs/m3_metrics.json`, `logs/m3_model_selection.csv` and `data/m3/predictions.parquet`.

**Framing.** For each labelled entry, rank the `n` references of its paper and keep the top 5 (learning
to rank, one query per entry). Three scorers, all classical:

| Scorer | What it learns |
|---|---|
| `baseline` | nothing: `10·f_arxiv_eq + 3·f_doi_eq + f_title_tfidf_char` |
| `logreg` | pointwise: `StandardScaler` + `LogisticRegression(class_weight="balanced")`, NaN filled with the train median plus a `*_missing` flag |
| `ranker` | listwise: `LGBMRanker(objective="lambdarank")`, one group per entry, NaN kept (fallback `HistGradientBoostingClassifier` without lightgbm) |

**Model choice.** Every configuration gets a 5-fold `GroupKFold` by publication on train (no publication in
both folds) and the MRR@5 on valid; the chosen one has the best mean of the two (two valid papers alone
are too noisy). Train rows of `predictions` are **out-of-fold**: each train paper is ranked by a model
that never saw it.

**Post-processing** (each can be switched off for the ablation):
1. `arxiv_rule`: a cell with `f_arxiv_eq = 1` is always rank 1;
2. `assignment`: inside each paper, a one-to-one assignment (Hungarian, `linear_sum_assignment`) on the
   paper-normalized scores; the assigned reference moves to rank 1 when its score is at least `tau ×` the
   best score of its row (`tau` tuned on valid).

| Section | Functions |
|---|---|
| 1. Scorers | `Baseline`, `LogReg`, `Ranker`, `make_scorer`, `split_frames`, `score_pairs` |
| 2. Cross-validation and choice | `oof_scores`, `select_model` |
| 3. Post-processing | `post_arxiv_rule`, `post_assignment`, `postprocess`, `tune_tau` |
| 4. Evaluation | `evaluate`, `evaluate_by`, `nil_report` (`mrr_at_k`, `hit_at_k`, `rank_candidates` of 22) |
| 5. `predictions` | `predictions_table`, `check_predictions` |
| 6. Ablation | `ablation`, `leave_one_publication_out` |
| 7. Full run | `run_model` (called by `25_run_m3`) |
| 8. Tests and dev demo | `ModelTests` |

In [ ]:
if "RUN_TEST" not in globals():
    RUN_TEST = True
_RUN_TEST_23 = RUN_TEST
if "FEATURES" not in globals():
    RUN_TEST = False
    get_ipython().run_line_magic("run", "-i ./22_features.ipynb")
RUN_TEST = _RUN_TEST_23

import copy
from scipy.optimize import linear_sum_assignment
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

try:
    import lightgbm as lgb
    HAS_LIGHTGBM = True
except ImportError:   # the ranker falls back to HistGradientBoostingClassifier
    HAS_LIGHTGBM = False

METRICS_PATH = LOG_DIR / "m3_metrics.json"
SELECTION_PATH = LOG_DIR / "m3_model_selection.csv"
PREDICTIONS_PATH = M3_DIR / "predictions.parquet"
PREDICTION_COLUMNS = ["arxiv_id", "partition", "citation_key", "groundtruth_arxiv_id",
                      "cand_1", "cand_2", "cand_3", "cand_4", "cand_5"]
PARTITIONS = ("train", "valid", "test")
K = 5
N_FOLDS = 5
MODEL_GRID = (   # (kind, params) tried by select_model
    [("baseline", {})]
    + [("logreg", {"C": c, "manual_weight": w}) for c in (0.1, 1.0, 10.0) for w in (1, 3, 5)]
    + [("ranker", {"num_leaves": leaves, "manual_weight": w}) for leaves in (15, 31) for w in (1, 3)]
)
TAU_GRID = (0.5, 0.6, 0.7, 0.8, 0.9, 0.95)
NIL_MODE = "zero"   # main MRR mode until the teacher answers; both are always reported
print(f"23 loaded | lightgbm: {HAS_LIGHTGBM} | {len(MODEL_GRID)} configurations")

## 1. Scorers

Each scorer has `fit(train, valid=None)` and `score(pairs) -> np.ndarray` aligned with the rows of
`pairs`. `features` defaults to every `f_*` column that has at least one value in train.
Manual entries weigh `manual_weight` times an automatic one (the manual labels are the ground truth the
automatic matcher is measured against; there are far fewer of them).

In [ ]:
def split_frames(pairs):
    """(train, valid, test) by the partition column."""
    return tuple(pairs[pairs["partition"] == p] for p in PARTITIONS)


def _usable_features(train, features=None):
    features = list(features or FEATURES)
    return [f for f in features if f in train.columns and train[f].notna().any()]


def _weights(frame, manual_weight):
    return np.where(frame["source"].astype(str).to_numpy() == "manual", float(manual_weight), 1.0)


class Baseline:
    """Not learned: identifiers first, then the character TF-IDF of the titles."""
    kind = "baseline"

    def __init__(self, features=None, **_):
        self.features = features

    def fit(self, train, valid=None):
        return self

    def score(self, pairs):
        def col(name):
            if self.features is not None and name not in self.features:
                return 0.0
            return np.nan_to_num(pairs[name].to_numpy(dtype=np.float64), nan=0.0)
        return 10 * col("f_arxiv_eq") + 3 * col("f_doi_eq") + col("f_title_tfidf_char")


class LogReg:
    """Pointwise logistic regression; NaN -> train median + *_missing flag."""
    kind = "logreg"

    def __init__(self, C=1.0, manual_weight=1, features=None, **_):
        self.C, self.manual_weight, self.features = C, manual_weight, features

    def _matrix(self, frame):
        x = frame[self.features_].to_numpy(dtype=np.float64)
        miss = np.isnan(x[:, self.flagged_])
        x = np.where(np.isnan(x), self.medians_, x)
        return np.hstack([x, miss.astype(np.float64)])

    def fit(self, train, valid=None):
        self.features_ = _usable_features(train, self.features)
        x = train[self.features_].to_numpy(dtype=np.float64)
        self.medians_ = np.nan_to_num(np.nanmedian(x, axis=0), nan=0.0)
        self.flagged_ = np.where(np.isnan(x).any(axis=0))[0]
        self.scaler_ = StandardScaler().fit(self._matrix(train))
        self.model_ = LogisticRegression(C=self.C, class_weight="balanced", max_iter=2000, random_state=SEED)
        self.model_.fit(self.scaler_.transform(self._matrix(train)), train["label"].to_numpy(),
                        sample_weight=_weights(train, self.manual_weight))
        return self

    def score(self, pairs):
        return self.model_.decision_function(self.scaler_.transform(self._matrix(pairs)))

    def coefficients(self):
        names = self.features_ + [f"{self.features_[i]}_missing" for i in self.flagged_]
        return pd.Series(self.model_.coef_[0], index=names).sort_values(key=np.abs, ascending=False)


class Ranker:
    """LGBMRanker (lambdarank), one group per entry; early stopping on valid when it has positives."""
    kind = "ranker"

    def __init__(self, num_leaves=31, manual_weight=1, learning_rate=0.05, n_estimators=500,
                 min_child_samples=20, features=None, **_):
        self.params = dict(num_leaves=num_leaves, learning_rate=learning_rate, n_estimators=n_estimators,
                           min_child_samples=min_child_samples)
        self.manual_weight, self.features = manual_weight, features

    @staticmethod
    def _grouped(frame):
        frame = frame.sort_values(KEY, kind="mergesort")
        return frame, frame.groupby(KEY, sort=False).size().to_numpy()

    def fit(self, train, valid=None):
        self.features_ = _usable_features(train, self.features)
        train, groups = self._grouped(train)
        y, w = train["label"].to_numpy(dtype=int), _weights(train, self.manual_weight)
        if not HAS_LIGHTGBM:
            self.model_ = HistGradientBoostingClassifier(max_leaf_nodes=self.params["num_leaves"],
                                                         learning_rate=self.params["learning_rate"],
                                                         max_iter=self.params["n_estimators"], random_state=SEED)
            self.model_.fit(train[self.features_], y, sample_weight=w)
            return self
        self.model_ = lgb.LGBMRanker(objective="lambdarank", random_state=SEED, verbose=-1, **self.params)
        fit_args = {}
        if valid is not None and len(valid) and valid["label"].sum() > 0:
            valid, v_groups = self._grouped(valid)
            fit_args = dict(eval_set=[(valid[self.features_], valid["label"].to_numpy(dtype=int))],
                            eval_group=[v_groups], eval_at=[K],
                            callbacks=[lgb.early_stopping(50, verbose=False)])
        # LightGBM weighs a query by the weights of its rows; they are equal inside an entry.
        # eval_set is deprecated in lightgbm 4.7 but is the only spelling older versions (Colab) accept.
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=FutureWarning)
            warnings.filterwarnings("ignore", message=".*eval_set.*")
            self.model_.fit(train[self.features_], y, group=groups, sample_weight=w, **fit_args)
        return self

    def score(self, pairs):
        if not HAS_LIGHTGBM:
            return self.model_.predict_proba(pairs[self.features_])[:, 1]
        return self.model_.predict(pairs[self.features_])

    def importances(self):
        if not HAS_LIGHTGBM:
            return pd.Series(dtype=float)
        gain = self.model_.booster_.feature_importance("gain")
        return pd.Series(gain, index=self.features_).sort_values(ascending=False)


SCORERS = {"baseline": Baseline, "logreg": LogReg, "ranker": Ranker}


def make_scorer(kind, params=None, features=None):
    return SCORERS[kind](**{**(params or {}), "features": features})


def score_pairs(model, pairs):
    """Scores of a fitted scorer as a Series indexed like pairs."""
    return pd.Series(model.score(pairs), index=pairs.index, dtype="float64")


def config_name(kind, params):
    return kind + ("" if not params else "(" + ", ".join(f"{k}={v}" for k, v in params.items()) + ")")

## 2. Grouped cross-validation and model choice

In [ ]:
def oof_scores(kind, params, train, valid=None, features=None, n_folds=N_FOLDS):
    """Out-of-fold scores on train (GroupKFold by paper) and the MRR@5 of each fold."""
    scores = pd.Series(np.nan, index=train.index, dtype="float64")
    papers = train["arxiv_id"].astype(str).to_numpy()
    folds = min(n_folds, len(set(papers)))
    fold_mrr = []
    if folds < 2:
        return scores, fold_mrr
    for fit_idx, held_idx in GroupKFold(n_splits=folds).split(train, groups=papers):
        fit_part, held = train.iloc[fit_idx], train.iloc[held_idx]
        model = make_scorer(kind, params, features).fit(fit_part, valid)
        scores.iloc[held_idx] = model.score(held)
        fold_mrr.append(mrr_at_k(rank_candidates(held, scores.iloc[held_idx], K), gold_of(held), K, NIL_MODE))
    return scores, fold_mrr


def select_model(pairs, grid=MODEL_GRID, features=None, n_folds=N_FOLDS, verbose=True):
    """One row per configuration: grouped-CV MRR@5 on train, MRR@5 on valid, and their mean (the choice)."""
    train, valid, _ = split_frames(pairs)
    rows = []
    for kind, params in grid:
        if kind == "ranker" and not HAS_LIGHTGBM and params.get("manual_weight", 1) != 1:
            continue
        t0 = time.time()
        _, fold_mrr = oof_scores(kind, params, train, valid, features, n_folds)
        model = make_scorer(kind, params, features).fit(train, valid)
        valid_mrr = (mrr_at_k(rank_candidates(valid, model.score(valid), K), gold_of(valid), K, NIL_MODE)
                     if len(valid) else np.nan)
        cv = float(np.mean(fold_mrr)) if fold_mrr else np.nan
        rows.append({"config": config_name(kind, params), "kind": kind, "params": json.dumps(params),
                     "cv_mrr5": cv, "cv_std": float(np.std(fold_mrr)) if fold_mrr else np.nan,
                     "valid_mrr5": valid_mrr, "choice_score": np.nanmean([cv, valid_mrr]),
                     "seconds": round(time.time() - t0, 1)})
        if verbose:
            print(f"  {rows[-1]['config']:<45} cv {cv:.4f}  valid {valid_mrr:.4f}  ({rows[-1]['seconds']}s)")
    table = pd.DataFrame(rows).sort_values("choice_score", ascending=False, kind="mergesort").reset_index(drop=True)
    return table

## 3. Post-processing

Scores are first normalized per paper to `[0, 1]` (`(s − paper min) / (paper max − paper min)`), so `tau`
means the same thing for every scorer; an assigned reference is promoted when its normalized score is at
least `tau ×` the best normalized score of its row. The assignment maximizes the total normalized score of the paper with
each reference used at most once (a reference is cited by at most one entry of a bibliography, after
M2 merged duplicate entries); `m > n` is allowed, the extra entries keep their own ranking.

In [ ]:
_BOOST = 1e6


def _paper_normalized(pairs, scores):
    """(s - paper min) / (paper max - paper min): one scale for every scorer, rows stay comparable."""
    s = pd.Series(np.asarray(scores, dtype=np.float64), index=pairs.index)
    s = s.fillna(s.min() if s.notna().any() else 0.0)
    g = s.groupby(pairs["arxiv_id"])
    lo, hi = g.transform("min"), g.transform("max")
    return ((s - lo) / (hi - lo).replace(0, 1)).where(hi > lo, 1.0)


def post_arxiv_rule(pairs, scores):
    """An equal arXiv id always ranks first."""
    s = np.asarray(scores, dtype=np.float64).copy()
    s[pairs["f_arxiv_eq"].to_numpy() == 1] += _BOOST
    return s


def post_assignment(pairs, scores, tau=0.9):
    """Hungarian assignment per paper; promote the assigned cell if it is >= tau x the best of its row."""
    s = np.asarray(scores, dtype=np.float64).copy()
    norm = _paper_normalized(pairs, s).to_numpy()
    for _, cells in pairs.groupby("arxiv_id", sort=False).indices.items():
        sub = pairs.iloc[cells]
        rows, i = np.unique(sub["citation_key"].to_numpy(), return_inverse=True)
        cols, j = np.unique(sub["ref_arxiv_id"].to_numpy(), return_inverse=True)
        matrix = np.zeros((len(rows), len(cols)))
        matrix[i, j] = norm[cells]
        r_idx, c_idx = linear_sum_assignment(-matrix)
        chosen = set(zip(r_idx, c_idx))
        row_best = matrix.max(axis=1)[i]
        promote = np.array([(a, b) in chosen for a, b in zip(i, j)]) & (norm[cells] >= tau * row_best)
        s[cells[promote]] += _BOOST / 2      # below the arXiv-rule boost
    return s


def postprocess(pairs, scores, arxiv_rule=True, assignment=True, tau=0.9):
    s = np.asarray(scores, dtype=np.float64)
    if assignment and tau is not None:
        s = post_assignment(pairs, s, tau)
    if arxiv_rule:
        s = post_arxiv_rule(pairs, s)
    return s


def tune_tau(pairs, scores, grid=TAU_GRID):
    """Best tau on these (valid) pairs; None (no assignment) if it never helps."""
    gold = gold_of(pairs)
    base = mrr_at_k(rank_candidates(pairs, postprocess(pairs, scores, assignment=False), K), gold, K, NIL_MODE)
    best, best_tau = base, None
    for tau in grid:
        value = mrr_at_k(rank_candidates(pairs, postprocess(pairs, scores, tau=tau), K), gold, K, NIL_MODE)
        if value > best + 1e-12:
            best, best_tau = value, tau
    return best_tau, {"no_assignment": base, "best": best}

## 4. Evaluation

`evaluate` returns MRR@5 in both nil modes, Hit@1, Hit@5 and the entry counts. `evaluate_by` splits it by
partition, label source (`manual` / `auto`) and `bib_source` (`bib` / `bibitem`, when the cleaned entries
are given). `nil_report` measures how well a threshold on the best score of an entry detects entries with
**no** true reference (precision and recall of "no match").

In [ ]:
def evaluate(pairs, scores, k=K):
    ranked, gold = rank_candidates(pairs, scores, k), gold_of(pairs)
    return {"mrr5_zero": mrr_at_k(ranked, gold, k, "zero"), "mrr5_skip": mrr_at_k(ranked, gold, k, "skip"),
            "hit1": hit_at_k(ranked, gold, 1), "hit5": hit_at_k(ranked, gold, k),
            "n_entries": len(gold), "n_with_true_ref": sum(v is not None for v in gold.values()),
            "n_papers": int(pairs["arxiv_id"].nunique())}


def evaluate_by(pairs, scores, entries_c=None):
    """{partition: {"all": metrics, "manual": ..., "auto": ..., "bib": ..., "bibitem": ...}}."""
    frame = pairs.assign(_s=np.asarray(scores, dtype=np.float64))
    if entries_c is not None and "bib_source" in entries_c:
        frame = frame.merge(entries_c[KEY + ["bib_source"]], on=KEY, how="left")
    out = {}
    for partition in PARTITIONS:
        part = frame[frame["partition"] == partition]
        if part.empty:
            continue
        out[partition] = {"all": evaluate(part, part["_s"])}
        for column in ("source", "bib_source"):
            if column in part:
                for value, sub in part.groupby(column, observed=True):
                    out[partition][str(value)] = evaluate(sub, sub["_s"])
    return out


def nil_report(pairs, scores, thresholds=None):
    """Precision / recall of 'this entry has no true reference' when its best score < threshold."""
    frame = pairs.assign(_s=np.asarray(scores, dtype=np.float64))
    per_entry = frame.groupby(KEY).agg(best=("_s", "max"), has_true=("label", "max"))
    truth = per_entry["has_true"] == 0
    if thresholds is None:
        thresholds = np.unique(np.quantile(per_entry["best"], np.linspace(0.05, 0.95, 19)))
    rows = []
    for t in thresholds:
        said = per_entry["best"] < t
        tp = int((said & truth).sum())
        rows.append({"threshold": float(t), "precision": tp / max(int(said.sum()), 1),
                     "recall": tp / max(int(truth.sum()), 1), "share_said_nil": float(said.mean())})
    return pd.DataFrame(rows)

## 5. The `predictions` table

One row per labelled entry of every train / valid / test publication, with exactly the columns of the
statement. `cand_k` is null when the paper has fewer than `k` references; `groundtruth_arxiv_id` is null
when the entry has no true reference. `check_predictions` stops with the list of every broken rule.

In [ ]:
def predictions_table(ranked, gold, partition_of, k=K):
    """ranked: {(arxiv_id, key): [ref, ...]}; gold: {(arxiv_id, key): ref | None}; partition_of: {arxiv_id: p}."""
    rows = []
    for (arxiv_id, key), true_ref in gold.items():
        cands = list(ranked.get((arxiv_id, key), []))[:k]
        rows.append([arxiv_id, partition_of[arxiv_id], key, true_ref] + cands + [None] * (k - len(cands)))
    df = pd.DataFrame(rows, columns=PREDICTION_COLUMNS)
    for column in PREDICTION_COLUMNS:
        df[column] = df[column].astype("string")
    return df.sort_values(["partition", "arxiv_id", "citation_key"], kind="mergesort").reset_index(drop=True)


def check_predictions(df, refs=None, expected_keys=None, split_sizes=(("test", 2), ("valid", 2))):
    """Raise ValueError listing every problem; return True when the table is valid."""
    problems = []
    if list(df.columns) != PREDICTION_COLUMNS:
        problems.append(f"columns {list(df.columns)} != {PREDICTION_COLUMNS}")
    else:
        if not all(pd.api.types.is_string_dtype(df[c]) for c in PREDICTION_COLUMNS):
            problems.append("every column must be a string column")
        dup = df.duplicated(KEY)
        if dup.any():
            problems.append(f"{int(dup.sum())} duplicate (arxiv_id, citation_key): {df[dup][KEY].head(5).values.tolist()}")
        bad_part = set(df["partition"].dropna()) - set(PARTITIONS)
        if bad_part or df["partition"].isna().any():
            problems.append(f"partitions outside {PARTITIONS}: {sorted(bad_part)}")
        if df[["arxiv_id", "citation_key"]].isna().any().any():
            problems.append("null arxiv_id or citation_key")
        for name, size in split_sizes or ():
            n = df[df["partition"] == name]["arxiv_id"].nunique()
            if n != size:
                problems.append(f"{name} has {n} publications, expected {size}")
        if refs is not None:
            known = set(zip(refs["arxiv_id"].astype(str), refs["ref_arxiv_id"].astype(str)))
            for c in [f"cand_{i}" for i in range(1, K + 1)] + ["groundtruth_arxiv_id"]:
                cells = df[df[c].notna()]
                foreign = [(a, r) for a, r in zip(cells["arxiv_id"], cells[c]) if (a, r) not in known]
                if foreign:
                    problems.append(f"{c}: {len(foreign)} references not in their paper, e.g. {foreign[:3]}")
        cands = df[[f"cand_{i}" for i in range(1, K + 1)]]
        filled = cands.notna().to_numpy()
        if (filled[:, 1:] & ~filled[:, :-1]).any():
            problems.append("a cand_k is filled after an empty cand_(k-1)")
        repeated = sum(len(set(r.dropna())) != r.notna().sum() for _, r in cands.iterrows())
        if repeated:
            problems.append(f"{repeated} rows repeat a candidate")
        if expected_keys is not None:
            have = set(zip(df["arxiv_id"], df["citation_key"]))
            missing = set(expected_keys) - have
            extra = have - set(expected_keys)
            if missing:
                problems.append(f"{len(missing)} labelled entries missing, e.g. {sorted(missing)[:3]}")
            if extra:
                problems.append(f"{len(extra)} rows that are not labelled entries, e.g. {sorted(extra)[:3]}")
    if problems:
        raise ValueError("predictions table:\n- " + "\n- ".join(problems))
    return True

## 6. Ablation

Each variant is refitted with the chosen configuration and evaluated on valid and test (MRR@5 in the
main nil mode): the full model; without identifier features (and without the arXiv rule) to see whether
the model still works on entries that do not give an id; without the M2 hierarchy features; without
post-processing; with the title features only. `leave_one_publication_out` holds out one manual
publication at a time (training on every other labelled publication outside test) and gives the mean and
standard deviation of its MRR@5, a less noisy estimate than two test papers.

In [ ]:
ABLATIONS = {
    "full": {},
    "no_identifiers": {"drop": ["f_arxiv_eq", "f_doi_eq", "f_has_arxiv_hint"], "arxiv_rule": False},
    "no_hierarchy": {"drop": ["f_ctx_tfidf", "f_n_cites"]},
    "no_postprocessing": {"arxiv_rule": False, "assignment": False},
    "title_only": {"keep": FEATURE_GROUPS["title"]},
}


def _fit_and_score(kind, params, features, train, valid, frames, post):
    model = make_scorer(kind, params, features).fit(train, valid)
    return {name: postprocess(frame, model.score(frame), **post) for name, frame in frames.items()}


def ablation(pairs, kind, params, tau=None):
    train, valid, test = split_frames(pairs)
    out = {}
    for name, spec in ABLATIONS.items():
        features = [f for f in (spec.get("keep") or FEATURES) if f not in spec.get("drop", [])]
        post = {"arxiv_rule": spec.get("arxiv_rule", True), "assignment": spec.get("assignment", True) and tau is not None,
                "tau": tau}
        scored = _fit_and_score(kind, params, features, train, valid, {"valid": valid, "test": test}, post)
        out[name] = {p: (mrr_at_k(rank_candidates(f, scored[p], K), gold_of(f), K, NIL_MODE) if len(f) else None)
                     for p, f in (("valid", valid), ("test", test))}
    return out


def leave_one_publication_out(pairs, kind, params, tau=None):
    pool = pairs[pairs["partition"] != "test"]
    manual = sorted(pool[pool["source"] == "manual"]["arxiv_id"].astype(str).unique())
    scores = {}
    for arxiv_id in manual:
        held = pool[pool["arxiv_id"] == arxiv_id]
        rest = pool[pool["arxiv_id"] != arxiv_id]
        model = make_scorer(kind, params).fit(rest)
        s = postprocess(held, model.score(held), assignment=tau is not None, tau=tau)
        scores[arxiv_id] = mrr_at_k(rank_candidates(held, s, K), gold_of(held), K, NIL_MODE)
    values = list(scores.values())
    return {"n": len(values), "mean": float(np.mean(values)) if values else None,
            "std": float(np.std(values)) if values else None, "per_paper": scores}

## 7. Full run (`25_run_m3`)

`run_model()` reads `pairs.parquet`, chooses the configuration, refits it on train (early stopping on
valid for the ranker), takes out-of-fold scores for train, tunes `tau` on valid, evaluates every
partition, runs the ablation and writes `predictions.parquet` (after `check_predictions`),
`m3_metrics.json` and the selection table. The model is frozen by writing `chosen` into the metrics.

In [ ]:
def run_model(pairs=None, entries_c=None, refs=None, grid=MODEL_GRID, save=True, out_dir=None,
              do_ablation=True, split_sizes=(("test", 2), ("valid", 2))):
    t0 = time.time()
    pairs = pairs if pairs is not None else pd.read_parquet(PAIRS_PATH)
    pairs = pairs.copy()
    for column in ("arxiv_id", "source", "partition", "citation_key", "ref_arxiv_id"):
        pairs[column] = pairs[column].astype(str)
    train, valid, test = split_frames(pairs)
    print(f"[23] pairs: {len(pairs):,} | entries per partition: "
          f"{pairs.drop_duplicates(KEY)['partition'].value_counts().to_dict()}")

    selection = select_model(pairs, grid)
    best = selection.iloc[0]
    kind, params = best["kind"], json.loads(best["params"])
    print(f"[23] chosen: {best['config']} (cv {best['cv_mrr5']:.4f}, valid {best['valid_mrr5']:.4f})")

    model = make_scorer(kind, params).fit(train, valid)
    raw = pd.Series(np.nan, index=pairs.index, dtype="float64")
    raw.loc[train.index], _ = oof_scores(kind, params, train, valid)
    if raw.loc[train.index].isna().any():              # fewer than 2 train papers: in-sample scores
        raw.loc[train.index] = model.score(train)
    for frame in (valid, test):
        if len(frame):
            raw.loc[frame.index] = model.score(frame)
    tau, tau_info = tune_tau(valid, raw.loc[valid.index]) if len(valid) else (None, {})
    final = pd.Series(postprocess(pairs, raw, assignment=tau is not None, tau=tau), index=pairs.index)

    metrics = {
        "nil_mode_main": NIL_MODE, "k": K, "chosen": {"config": best["config"], "kind": kind, "params": params,
                                                     "tau": tau, "tau_tuning": tau_info},
        "models": {}, "final": evaluate_by(pairs, final, entries_c),
        "nil_detection_valid": nil_report(valid, raw.loc[valid.index]).to_dict("records") if len(valid) else [],
    }
    for kind_i in ("baseline", "logreg", "ranker"):     # the best configuration of each scorer, raw scores
        rows = selection[selection["kind"] == kind_i]
        if rows.empty:
            continue
        p = json.loads(rows.iloc[0]["params"])
        m = make_scorer(kind_i, p).fit(train, valid)
        metrics["models"][rows.iloc[0]["config"]] = {
            part: evaluate(f, m.score(f)) for part, f in (("valid", valid), ("test", test)) if len(f)}
    if hasattr(model, "importances"):
        metrics["feature_importance_gain"] = model.importances().round(2).to_dict()
    if hasattr(model, "coefficients"):
        metrics["logreg_coefficients"] = model.coefficients().round(4).to_dict()
    if do_ablation:
        metrics["ablation"] = ablation(pairs, kind, params, tau)
        metrics["leave_one_publication_out"] = leave_one_publication_out(pairs, kind, params, tau)

    ranked, gold = rank_candidates(pairs, final, K), gold_of(pairs)
    partition_of = dict(zip(pairs["arxiv_id"], pairs["partition"]))
    predictions = predictions_table(ranked, gold, partition_of)
    check_predictions(predictions, refs if refs is not None else pairs[["arxiv_id", "ref_arxiv_id"]].drop_duplicates(),
                      expected_keys=gold.keys(), split_sizes=split_sizes)
    metrics["seconds"] = round(time.time() - t0, 1)
    if save:
        out_dir = Path(out_dir) if out_dir else None
        pred_path = (out_dir / "predictions.parquet") if out_dir else PREDICTIONS_PATH
        metrics_path = (out_dir / "m3_metrics.json") if out_dir else METRICS_PATH
        selection_path = (out_dir / "m3_model_selection.csv") if out_dir else SELECTION_PATH
        for path in (pred_path, metrics_path, selection_path):
            path.parent.mkdir(parents=True, exist_ok=True)
        predictions.to_parquet(pred_path, index=False)
        metrics_path.write_text(json.dumps(metrics, indent=1, default=float), encoding="utf-8")
        selection.to_csv(selection_path, index=False)
        print(f"[23] wrote {pred_path} ({len(predictions):,} rows), {metrics_path}, {selection_path}")
    print(f"[23] done in {metrics['seconds']}s | final MRR@5 ({NIL_MODE}): "
          + ", ".join(f"{p} {v['all'][f'mrr5_{NIL_MODE}']:.4f}" for p, v in metrics["final"].items()))
    return {"metrics": metrics, "selection": selection, "predictions": predictions, "model": model, "scores": final}

## 8. Tests and dev demo

The toy data are built through the whole of `22` (cleaning, pairs, labels, features) from random titles,
so the models see realistic features. The dev demo runs `run_model` on the 31-paper dev sample with weak
labels and a random split (2 test, 2 valid papers): it checks the chain, its numbers are not results.

In [ ]:
import unittest

_WORDS = ("graph neural network learning deep model quantum field theory lattice spin glass random matrix "
          "transformer attention language vision protein folding dark matter black hole entropy optimal "
          "transport diffusion kernel bayesian inference sparse signal recovery stochastic gradient").split()


def _toy_pairs(n_papers=12, n_refs=8, n_entries=10, seed=SEED, with_hint=0.3):
    """Labelled pairs with features built by 22 from random titles: most entries cite one reference."""
    rng = np.random.default_rng(seed)
    entries, refs, labels = [], [], []
    for p in range(n_papers):
        arxiv_id = f"2601.{p:05d}"
        titles = [" ".join(rng.choice(_WORDS, size=6, replace=False)) for _ in range(n_refs)]
        authors = [[f"A. {rng.choice(_WORDS).title()}x", f"B. {rng.choice(_WORDS).title()}y"] for _ in range(n_refs)]
        for j in range(n_refs):
            refs.append(_ref(arxiv_id, f"2301.{p:03d}{j:02d}", titles[j], authors[j], year=2020))
        for i in range(n_entries):
            key = f"k{i}"
            if i < n_refs - 1:                      # cites reference i with a noisy title
                words = titles[i].split()
                words[rng.integers(len(words))] = str(rng.choice(_WORDS))
                hint = f"https://arxiv.org/abs/2301.{p:03d}{i:02d}" if rng.random() < with_hint else None
                entries.append(_entry(arxiv_id, key, " ".join(words), authors[i], year=2020 + int(rng.integers(0, 2)), url=hint))
                labels.append((arxiv_id, key, f"2301.{p:03d}{i:02d}"))
            else:                                   # cites something outside the list
                entries.append(_entry(arxiv_id, key, " ".join(rng.choice(_WORDS, size=5, replace=False)), ["Z. Nobody"]))
                labels.append((arxiv_id, key, None))
    e, r = clean_entries(pd.DataFrame(entries)), clean_refs(pd.DataFrame(refs))
    lab = pd.DataFrame(labels, columns=CELL).assign(source=["manual" if int(a[-2:]) < 4 else "auto" for a, _, _ in labels])
    ids = sorted(e["arxiv_id"].unique())
    split = {ids[0]: "test", ids[1]: "test", ids[2]: "valid", ids[3]: "valid"}
    pairs = add_features(attach_labels(build_pairs(e, r, verbose=False), lab, split), e, r, verbose=False)
    return pairs, e, r


class ModelTests(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        cls.pairs, cls.e, cls.r = _toy_pairs()

    def test_mrr_hand_example_both_modes(self):
        ranked = {("p", "e1"): ["r1"], ("p", "e2"): ["x", "y", "r2"], ("p", "e3"): list("abcde") + ["r3"], ("p", "e4"): []}
        gold = {("p", "e1"): "r1", ("p", "e2"): "r2", ("p", "e3"): "r3", ("p", "e4"): None}
        self.assertEqual(round(mrr_at_k(ranked, gold, 5, "zero"), 4), 0.3333)
        self.assertEqual(round(mrr_at_k(ranked, gold, 5, "skip"), 4), 0.4444)

    def test_models_rank_the_positive_first(self):
        train, valid, test = split_frames(self.pairs)
        for kind, params in (("logreg", {"C": 1.0}), ("ranker", {"num_leaves": 15, "min_child_samples": 5})):
            model = make_scorer(kind, params).fit(train, valid)
            result = evaluate(test, model.score(test))
            self.assertGreaterEqual(result["hit1"], 0.9, kind)
            self.assertGreater(result["mrr5_skip"], 0.9, kind)

    def test_oof_scores_cover_train_and_never_see_the_paper(self):
        train = split_frames(self.pairs)[0]
        scores, folds = oof_scores("logreg", {"C": 1.0}, train, n_folds=4)
        self.assertFalse(scores.isna().any())
        self.assertEqual(len(folds), 4)

    def test_arxiv_rule_forces_rank_one(self):
        p = pd.DataFrame({"arxiv_id": ["p"] * 3, "citation_key": ["e"] * 3, "ref_arxiv_id": ["a", "b", "c"],
                          "f_arxiv_eq": [0, 0, 1]})
        self.assertEqual(rank_candidates(p, post_arxiv_rule(p, [0.9, 0.5, -3.0]))[("p", "e")][0], "c")

    def test_assignment_resolves_a_shared_favourite(self):
        # e1 and e2 both prefer A; e1 is much surer, so e2 should get B (its close second).
        p = pd.DataFrame({"arxiv_id": ["p"] * 4, "citation_key": ["e1", "e1", "e2", "e2"],
                          "ref_arxiv_id": ["A", "B", "A", "B"], "f_arxiv_eq": [0] * 4})
        s = post_assignment(p, [0.95, 0.10, 0.80, 0.78], tau=0.5)
        ranked = rank_candidates(p, s)
        self.assertEqual(ranked[("p", "e1")][0], "A")
        self.assertEqual(ranked[("p", "e2")][0], "B")
        self.assertEqual(rank_candidates(p, post_assignment(p, [0.95, 0.10, 0.80, 0.10], tau=0.9))[("p", "e2")][0], "A")

    def test_predictions_table_shape_and_nulls(self):
        ranked = {("p1", "a"): ["r1", "r2"], ("p1", "b"): ["r2", "r1"]}
        gold = {("p1", "a"): "r1", ("p1", "b"): None}
        df = predictions_table(ranked, gold, {"p1": "test"})
        self.assertEqual(list(df.columns), PREDICTION_COLUMNS)
        self.assertTrue(df[["cand_3", "cand_4", "cand_5"]].isna().all().all())
        self.assertTrue(pd.isna(df.set_index("citation_key").loc["b", "groundtruth_arxiv_id"]))
        refs = pd.DataFrame({"arxiv_id": ["p1", "p1"], "ref_arxiv_id": ["r1", "r2"]})
        self.assertTrue(check_predictions(df, refs, expected_keys=gold.keys(), split_sizes=(("test", 1),)))

    def test_check_predictions_catches_errors(self):
        ranked = {("p1", "a"): ["r1", "r2"], ("p1", "b"): ["r2", "r1"]}
        gold = {("p1", "a"): "r1", ("p1", "b"): None}
        refs = pd.DataFrame({"arxiv_id": ["p1", "p1"], "ref_arxiv_id": ["r1", "r2"]})
        good = predictions_table(ranked, gold, {"p1": "test"})
        duplicated = pd.concat([good, good.iloc[[0]]], ignore_index=True)
        foreign = good.copy()
        foreign.loc[0, "cand_2"] = "r9"
        for broken, expected_keys in ((duplicated, gold.keys()), (foreign, gold.keys()),
                                      (good.iloc[[0]], gold.keys())):
            with self.assertRaises(ValueError):
                check_predictions(broken, refs, expected_keys=expected_keys, split_sizes=(("test", 1),))
        with self.assertRaises(ValueError):                    # 1 test paper where 2 are expected
            check_predictions(good, refs, expected_keys=gold.keys())

    def test_run_model_end_to_end_on_toy(self):
        grid = [("baseline", {}), ("logreg", {"C": 1.0, "manual_weight": 3}),
                ("ranker", {"num_leaves": 15, "min_child_samples": 5, "manual_weight": 1})]
        out = run_model(self.pairs, self.e, grid=grid, save=False)
        self.assertEqual(set(out["predictions"]["partition"]), {"train", "valid", "test"})
        self.assertEqual(len(out["predictions"]), self.pairs.drop_duplicates(KEY).shape[0])
        self.assertGreater(out["metrics"]["final"]["test"]["all"]["mrr5_skip"], 0.8)
        self.assertIn("no_identifiers", out["metrics"]["ablation"])


if RUN_TEST:
    _res = unittest.TextTestRunner(verbosity=1).run(unittest.defaultTestLoader.loadTestsFromTestCase(ModelTests))
    assert _res.wasSuccessful(), "23 tests failed"

In [ ]:
# Dev demo (weak labels, random split of the dev sample): checks the chain on real rows, not a result.
if RUN_TEST and DEV_BIBTEX.is_file():
    dev_entries, dev_refs = dev_frames()
    dev_e, dev_r = clean_entries(dev_entries), clean_refs(dev_refs)
    dev_lab = dev_labels(dev_e, dev_r)
    _papers = sorted(dev_lab["arxiv_id"].unique())
    _rng = np.random.default_rng(SEED)
    _pick = list(_rng.choice(_papers, size=4, replace=False))
    dev_split = {_pick[0]: "test", _pick[1]: "test", _pick[2]: "valid", _pick[3]: "valid"}
    dev_lab["source"] = ["manual" if a in _pick or i % 5 == 0 else "auto"
                         for i, a in enumerate(dev_lab["arxiv_id"])]
    dev_pairs = add_features(attach_labels(build_pairs(dev_e, dev_r), dev_lab, dev_split, refs=dev_r), dev_e, dev_r)
    dev_out = run_model(dev_pairs, dev_e, dev_r, out_dir=LOG_DIR / "m3_dev")
    display(dev_out["selection"])
    display(dev_out["predictions"].head(8))
    print(json.dumps({k: dev_out["metrics"][k] for k in ("chosen", "ablation")}, indent=1, default=float))